# Verify a Zambo receipt yourself

Every Zambo tool call returns an AER-1 receipt: the exact saved result and its SHA-256 digest. A receipt proves the saved result was not changed after it was saved. This notebook fetches a real receipt from the live stream and recomputes the digest with nothing but the Python standard library.

The standard behind the receipts: AER-1, IETF draft-zambo-aer1-26: https://datatracker.ietf.org/doc/draft-zambo-aer1/26/

In [ ]:
import json, hashlib, base64, urllib.request

def get(url):
    req = urllib.request.Request(url, headers={'User-Agent': 'aer1-notebook'})
    return json.load(urllib.request.urlopen(req, timeout=30))

recent = get('https://zambo.dev/api/receipts/recent')
receipt_id = recent[0]['id']
print('Latest receipt on the live stream:', receipt_id)

In [ ]:
receipt = get(f'https://zambo.dev/api/receipt/{receipt_id}')
print('Tool:', receipt['tools'], '| issued:', receipt['created_at'])
print('Claimed output_hash:', receipt['output_hash'])

In [ ]:
canonical = base64.b64decode(receipt['canonical_bytes'])
recomputed = 'sha256:' + hashlib.sha256(canonical).hexdigest()
print('Recomputed digest:  ', recomputed)
print('MATCH' if recomputed == receipt['output_hash'] else 'MISMATCH')

In [ ]:
status = get(f'https://zambo.dev/api/v2/receipt/{receipt_id}/verify')
print('Independent verifier says:', status.get('verification_status'))

corpus = get('https://zambo.dev/aer1/corpus.json')
print('AER-1 conformance corpus:', corpus['vector_count'], 'vectors,', corpus['check_count'], 'checks, kit', corpus['kit_version'])

## Try it yourself

Add Zambo to your AI: https://zambo.dev/add/ (endpoint https://zambo.dev/mcp, no API key, 20 free calls per day). Every call you make returns a receipt you can check exactly the way this notebook just did.